[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part2/ch6/ch6_ex_classification.ipynb)

In [ ]:
# 1. 문제정의
# 평가: f1
# target: STATUS
# 최종파일: result.csv(컬럼 1개 pred)

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd

# train = pd.read_csv("creditcard_train.csv")
# test = pd.read_csv("creditcard_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/creditcard_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/creditcard_test.csv")

# 3. 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print(train.shape, test.shape)

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum())

print("\n ===== 범주형 테이터 카테고리 =====")
cols = train.select_dtypes(include='object').columns
for col in cols:
    set_train = set(train[col])
    set_test= set(test[col])
    same = (set_train == set_test)
    if same:
        print(col, "\t카테고리 동일함")
    else:
        print(col, "\t카테고리 동일하지 않음")

print("\n ===== target 빈도 =====")
print(train['STATUS'].value_counts())

: 

In [1]:
# 4. 데이터 전처리
# 결측치 처리
print("삭제 전:",train.shape)
train.dropna(subset=['OCCUPATION_TYPE'], inplace=True)
print("삭제 후:",train.shape)

target = train.pop('STATUS')

# 원핫인코딩
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
score = f1_score(y_val, pred)
print('\n f1:', score)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

NameError: name 'train' is not defined

### 성능개선

In [30]:
# 2. 라이브러리 및 데이터 불러오기
import pandas as pd

# train = pd.read_csv("creditcard_train.csv")
# test = pd.read_csv("creditcard_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/creditcard_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/creditcard_test.csv")

# 4. 데이터 전처리
# 결측치 처리 (최빈값)
freq = train['OCCUPATION_TYPE'].mode()[0]
train['OCCUPATION_TYPE'] = train['OCCUPATION_TYPE'].fillna(freq)
test['OCCUPATION_TYPE'] = test['OCCUPATION_TYPE'].fillna(freq)

target = train.pop('STATUS')

# 스케일링 (성능 개선 효과 없음)
# from sklearn.preprocessing import RobustScaler
# scaler = RobustScaler()
# n_cols = train.select_dtypes(exclude='object').columns[:-1] # STATUS를 제외한 int, float
# train[n_cols] = scaler.fit_transform(train[n_cols])
# test[n_cols] = scaler.transform(test[n_cols])

# ID 제외
train = train.drop('ID', axis=1)
test = test.drop('ID', axis=1)

# 레이블 인코딩
from sklearn.preprocessing import LabelEncoder
cols = train.select_dtypes(include='object').columns
for col in cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
score = f1_score(y_val, pred)
print('f1:', score)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

f1: 0.3253968253968254
